# Study 2 — Linear KAN-FNO: Training & Symbolic Recovery

**What this notebook does:**
1. Clones `MiRoSi-52wab/Training` and installs the package
2. Loads `dataset_v3.h5` from Google Drive (or regenerates it on Colab's CPU)
3. Verifies the pipeline at exact initialization — `field_loss ≈ 0.0001%`
4. **Smoke test** — 2 epochs to confirm no NaNs, gradients flow, GPU memory is fine
5. Trains `KANTauTheta` (3 parameters) for 100 epochs
6. Plots training curves + γ_θ contractivity check
7. Runs symbolic recovery — confirms `ctrl ≈ [1, −1, 1]` (exact x²)

**Expected outcome:** The 3 learned control points converge to `[1, −1, 1]` to
3–4 decimal places, proving gradient descent recovers the correct symbolic form
of the bilinear double-contraction operator from FFT-generated data.

**Runtime estimate:** ~15–25 min on T4 GPU (100 epochs, B=16, K=140).

---
> **Before running:** `Runtime → Change runtime type → T4 GPU`

## §1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## §2 — Clone repository and install package

The repo root (`MiRoSi-52wab/Training`) contains `pyproject.toml` directly,
so after cloning to `/content/ls-kan-fno` that path is both the repo root
**and** the installable root — there is no sub-directory to descend into.

In [ ]:
GITHUB_REPO = "MiRoSi-52wab/Training"
CLONE_DIR   = "/content/ls-kan-fno"

import subprocess, sys, os

# Clone (idempotent: pulls latest if already present after a session restart)
result = subprocess.run(
    ["git", "clone", f"https://github.com/{GITHUB_REPO}.git", CLONE_DIR],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Cloned successfully.")
elif "already exists" in result.stderr:
    print("Repo already present — pulling latest changes.")
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
else:
    print(result.stderr)
    raise RuntimeError("git clone failed — check the repo name and your authentication.")

# Install the package (pyproject.toml is at the repo root, same as CLONE_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", CLONE_DIR, "-q"],
    check=True
)
print("Package installed.")

# Set working directory and Python path
REPO_ROOT = CLONE_DIR   # /content/ls-kan-fno
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(f"Working directory: {os.getcwd()}")

## §3 — Configure paths

Set `DATA_PATH` to where you placed `dataset_v3.h5` on your Drive,
and `RUN_DIR` to where checkpoints and logs should be saved.
**Edit this cell before running anything else.**

In [ ]:
# ── EDIT THESE TWO LINES ────────────────────────────────────────────────────
DATA_PATH = "/content/drive/MyDrive/ls_kan_fno/data/dataset_v3.h5"
RUN_DIR   = "/content/drive/MyDrive/ls_kan_fno/runs/linear_study2"
# ────────────────────────────────────────────────────────────────────────────

from pathlib import Path
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

data_ok = Path(DATA_PATH).exists()
print(f"DATA_PATH : {DATA_PATH}  {'✓ found' if data_ok else '✗ NOT FOUND — see §4'}")
print(f"RUN_DIR   : {RUN_DIR}  (created)")

## §4 — (Optional) Generate dataset_v3 on this machine

**Skip this cell if you already have `dataset_v3.h5` on Drive.**

If `DATA_PATH` was not found above, run this cell to regenerate the dataset
on Colab's CPU (~5 min) and save it directly to Drive.

In [ ]:
from pathlib import Path

if Path(DATA_PATH).exists():
    print(f"dataset_v3.h5 already exists at {DATA_PATH} — skipping generation.")
else:
    print(f"Generating dataset_v3.h5 → {DATA_PATH}")
    print("This takes ~5 min on CPU. Grab a coffee.")

    from generation.generate_dataset import generate
    from utils.config_loader import load_config

    cfg = load_config("configs/data.yaml")
    cfg["tag"]        = "v3"
    cfg["output_dir"] = str(Path(DATA_PATH).parent)
    generate(cfg)

    assert Path(DATA_PATH).exists(), "Generation finished but file not found — check output_dir."
    print(f"\ndataset_v3.h5 saved to {DATA_PATH}")

## §5 — GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {props.name}")
    print(f"VRAM : {props.total_memory / 1e9:.1f} GB")
    print(f"Torch: {torch.__version__}")
else:
    print("WARNING: No GPU detected.")
    print("Go to Runtime → Change runtime type → T4 GPU, then re-run from §1.")
    print("Training on CPU will work but takes ~10× longer.")

## §6 — Sanity check: field_loss at exact initialization

With `ctrl = [1, −1, 1]` (the exact x² representation), the LSFNO with K=140
is identical to running 140 fixed-point iterations at α₀=10.0. Since dataset_v3
was generated with the same α₀ and tol=1e-7, the model's output should match
the stored `eps_star` to within machine precision.

**Expected:** `field_loss < 0.001` (we've measured ~0.0001% on this dataset).
If this fails, something is wrong with the data path or the package installation.

In [ ]:
import torch
from utils.config_loader import load_config
from models.kan_tau_theta import KANTauTheta
from models.ls_fno import LSFNO
from datasets.micromechanics import DataLoaderFactory
from training.losses import field_loss

# Load config and patch paths
cfg = load_config("configs/training_colab.yaml")
cfg["train_path"] = DATA_PATH
cfg["val_path"]   = DATA_PATH
cfg["test_path"]  = DATA_PATH

# Build model with FROZEN exact control points
tau = KANTauTheta(R=1.0, shared=True, trainable=False, n_comp=3)
model = LSFNO.from_config(cfg, tau_theta=tau)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

loaders = DataLoaderFactory.from_config(cfg)
batch   = next(iter(loaders["val"]))
batch   = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

import time
with torch.no_grad():
    t0 = time.time()
    eps_pred = model(batch["C_field"], batch["eps_bar"], use_checkpointing=False)
    dt = time.time() - t0
    fl = field_loss(eps_pred, batch["eps_star"]).item()

print(f"field_loss at exact ctrl=[1,−1,1]: {fl:.6f}  ({fl*100:.4f}%)")
print(f"Forward pass time (B={batch['C_field'].shape[0]}, K=140): {dt:.2f}s")

assert fl < 0.001, f"Sanity check FAILED: field_loss={fl:.4f}. Check DATA_PATH and package install."
print("\nSanity check PASSED ✓  — pipeline is correctly wired.")

## §7 — Smoke test: 2 epochs with trainable KAN

This is the test that OOM-killed WSL2 locally (RSS grew to ~6 GB), but runs
fine on a T4 GPU with gradient checkpointing.

Checks:
- Loss is finite (no NaNs from float64 overflow)
- Gradient reaches `ctrl` — the 3 points start moving from `[1, −1, 1]`
- `γ_θ` stays below the contractivity bound `2/(κ+1) ≈ 0.182`
- Prints a per-epoch wall-clock estimate so you can plan the 100-epoch run

**Expected:** both epochs finish without error, `train_field` is a finite
positive number, `ctrl` shifts slightly from its starting value.

## §8 — Full training: 100 epochs

The only trainable parameters are the 3 B-spline control points `ctrl`.
Everything else (the Green operator, FFT steps, Voigt↔Mandel conversions)
is fixed analytic computation.

Epoch log columns:
- `train_field` / `val_field` — relative L2 strain field error
- `train_eff`   / `val_eff`   — relative directional-modulus error
- `gamma_theta_max`           — empirical Lipschitz constant of τ_θ (every 5 epochs)
- `gamma_theta_bound`         — theoretical limit 2/(κ+1) ≈ 0.182

Checkpoints saved to Drive every 10 epochs and on every new val-loss best.

## §7 — Train KANTauTheta (100 epochs)

The only trainable parameters are the 3 B-spline control points `ctrl`.
Everything else (the Green operator, FFT steps, Voigt↔Mandel conversions)
is fixed analytic computation.

Epoch log columns:
- `train_field` / `val_field` — relative L2 strain field error
- `train_eff`   / `val_eff`   — relative directional-modulus error
- `gamma_theta_max`           — empirical Lipschitz constant of τ_θ
- `gamma_theta_bound`         — theoretical limit 2/(κ+1) ≈ 0.182

Checkpoints are saved every 10 epochs and whenever validation loss improves.

## §9 — Training curves

Two panels:
- **Left**: field loss (log scale) — should decrease monotonically and plateau near the machine-precision floor (~0.0001%)
- **Right**: γ_θ contractivity check — must stay below the dashed red line 2/(κ+1) ≈ 0.182 throughout training

## §8 — Training curves

Two panels:
- **Left**: field loss (log scale) — should decrease monotonically and plateau near the machine-precision floor (~0.0001%)
- **Right**: γ_θ contractivity check — must stay below the dashed red line 2/(κ+1) ≈ 0.182 throughout training

## §10 — Inspect trained control points

Quick look at what `ctrl` converged to before running the full symbolic recovery.
Expected: `[1.xxx, −1.xxx, 1.xxx]` close to the exact `[1, −1, 1]`.

## §9 — Inspect trained control points

Quick look at what `ctrl` converged to before running the full symbolic recovery.
Expected: `[1.xxx, −1.xxx, 1.xxx]` close to the exact `[1, −1, 1]`.

## §11 — Symbolic recovery

Full pipeline:
1. Reconstruct the learned edge function φ(x) over x ∈ [−1, 1]
2. Compare to x² pointwise
3. Fit to a candidate library: x², x, |x|, const, x³, x⁴
4. Report R² for each candidate and give a PASS/FAIL verdict

**PASS criteria:**
- `max |ctrl − [1,−1,1]| < 0.01`
- `R²(x²) ≥ 0.9999`
- `x²` is the best-fitting candidate

## §10 — Symbolic recovery

Full pipeline:
1. Reconstruct the learned edge function φ(x) over x ∈ [−1, 1]
2. Compare to x² pointwise
3. Fit to a candidate library: x², x, |x|, const, x³, x⁴
4. Report R² for each candidate and give a PASS/FAIL verdict

**PASS criteria:**
- `max |ctrl − [1,−1,1]| < 0.01`
- `R²(x²) ≥ 0.9999`
- `x²` is the best-fitting candidate

## §12 — (Optional) LBFGS refinement

With only 3 parameters, Adam is usually sufficient. If `ctrl` hasn't converged
tightly to `[1, −1, 1]` after 100 epochs, a handful of LBFGS steps on a small
batch can close the gap.

**Only run this if the symbolic recovery in §11 did not PASS.**

## §11 — (Optional) LBFGS refinement

With only 3 parameters, Adam is usually sufficient. If `ctrl` hasn't converged
tightly to `[1, −1, 1]` after 100 epochs, a handful of LBFGS steps on a small
batch can close the gap.

**Only run this if the symbolic recovery in §10 did not PASS.**

In [ ]:
# Uncomment to run LBFGS refinement

# from utils.config_loader import load_config
# from training.trainer import Trainer
#
# cfg = load_config("configs/training_colab.yaml")
# cfg["train_path"]  = DATA_PATH
# cfg["val_path"]    = DATA_PATH
# cfg["test_path"]   = DATA_PATH
# cfg["output_dir"]  = RUN_DIR
# cfg["lbfgs_steps"] = 50        # more steps for tighter convergence
#
# # Reload trainer from best checkpoint and run LBFGS
# trainer = Trainer(cfg)
# final_loss = trainer.fit_lbfgs()
# print(f"LBFGS final loss: {final_loss:.6e}")
#
# # Re-run symbolic recovery
# from symbolic.recover import recover
# results = recover(str(Path(RUN_DIR) / "last_lbfgs_checkpoint.pt"), plot=True)

---
## Summary

| Quantity | Target | Measured |
|---|---|---|
| ctrl[0] | 1.0000 | *(fill after training)* |
| ctrl[1] | −1.0000 | *(fill after training)* |
| ctrl[2] | 1.0000 | *(fill after training)* |
| max δ_ctrl | < 0.01 | *(fill after recovery)* |
| R²(x²) | ≥ 0.9999 | *(fill after recovery)* |
| Best symbolic fit | x² | *(fill after recovery)* |
| val field loss (best) | < 0.001 | *(fill after training)* |
| γ_θ max | < 0.182 | *(fill after training)* |

**Next notebook:** `02_symbolic_recovery.ipynb` — deeper analysis of the trained
edges, per-component breakdown, and comparison against the Yarotsky-MLP baseline.